# Benchmarking frozen ESM-2 representations

**CSE 763 Advanced Bioinformatics / CSE 443 Bioinformatics — Term Project**

Four ESM-2 checkpoints (8M → 650M parameters) are used as frozen feature
extractors for a single downstream task: separating human transmembrane proteins
from soluble cytoplasmic ones. A logistic-regression probe is trained on the
mean-pooled embeddings, against two composition baselines and a
randomly-initialised encoder of identical architecture.

**Before you start:** switch the runtime to a GPU.
Runtime → Change runtime type → Hardware accelerator → **T4 GPU**.

Then use Runtime → **Run all**. The whole notebook takes roughly 25–35 minutes,
most of it spent embedding with the 650M checkpoint.


## 0. Environment

In [ ]:
!nvidia-smi || echo "NO GPU DETECTED - switch the runtime to T4 before continuing"


In [ ]:
%pip install -q "transformers>=4.40" "tabulate" "umap-learn" 2>/dev/null
import torch, transformers
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("transformers", transformers.__version__)


## 1. Write the pipeline modules

Each cell below writes one module to `src/`. Nothing is executed yet.

In [ ]:
import os, pathlib
pathlib.Path('src').mkdir(exist_ok=True)
print('working directory:', os.getcwd())


In [ ]:
%%writefile src/common.py
"""Shared configuration, paths, and helpers for the ESM-2 scaling benchmark."""
from __future__ import annotations

import json
import logging
import os
import random
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np

SEED = 42

ROOT = Path(__file__).resolve().parent.parent
DATA = ROOT / "data"
SPLITS = DATA / "splits"
EMB = DATA / "emb"
RESULTS = ROOT / "results"
FIGURES = ROOT / "figures"
REPORT = ROOT / "report"

for _d in (DATA, SPLITS, EMB, RESULTS, FIGURES, REPORT):
    _d.mkdir(parents=True, exist_ok=True)

AA = "ACDEFGHIKLMNPQRSTVWY"

# Sequence length window. ESM-2 was pretrained at 1024 tokens; we stay well inside it.
MIN_LEN = 50
MAX_LEN = 1000

# Redundancy reduction: drop a sequence if its 4-mer Jaccard similarity to an
# already-kept sequence exceeds this. ~0.5 is a rough proxy for high identity.
JACCARD_THRESHOLD = 0.5
KMER_K = 4

SPLIT_FRACTIONS = (0.70, 0.15, 0.15)  # train / val / test


@dataclass(frozen=True)
class ModelSpec:
    key: str            # short name used in filenames
    hf_id: str          # HuggingFace checkpoint id
    params: int         # approximate parameter count
    layers: int
    dim: int            # embedding dimensionality
    random_init: bool = False   # if True, do NOT load pretrained weights

    @property
    def label(self) -> str:
        if self.random_init:
            return "ESM-2 8M (random init)"
        return f"ESM-2 {self.params // 1_000_000}M"


MODELS: list[ModelSpec] = [
    ModelSpec("esm2_8m",   "facebook/esm2_t6_8M_UR50D",    8_000_000,  6,  320),
    ModelSpec("esm2_35m",  "facebook/esm2_t12_35M_UR50D",  35_000_000, 12, 480),
    ModelSpec("esm2_150m", "facebook/esm2_t30_150M_UR50D", 150_000_000, 30, 640),
    ModelSpec("esm2_650m", "facebook/esm2_t33_650M_UR50D", 650_000_000, 33, 1280),
    # Control: identical architecture to the 8M model but with randomly
    # initialised weights. Isolates the contribution of pretraining from the
    # contribution of the architecture + mean pooling.
    ModelSpec("rand_8m",   "facebook/esm2_t6_8M_UR50D",    8_000_000,  6,  320,
              random_init=True),
]

MODELS_BY_KEY = {m.key: m for m in MODELS}

# Pooling / layer variants stored by Stage 02 and compared in the ablation study.
VARIANTS = ["mean_last", "cls_last", "max_last", "mean_mid"]
VARIANT_LABELS = {
    "mean_last": "Mean pool, final layer",
    "cls_last": "<cls> token, final layer",
    "max_last": "Max pool, final layer",
    "mean_mid": "Mean pool, middle layer",
}

CLASS_NAMES = {0: "Soluble / cytoplasmic", 1: "Transmembrane"}


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


def get_logger(name: str) -> logging.Logger:
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)-7s | %(name)s | %(message)s",
        datefmt="%H:%M:%S",
    )
    return logging.getLogger(name)


def save_json(obj, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as fh:
        json.dump(obj, fh, indent=2, default=str)


def load_json(path: Path):
    with open(path) as fh:
        return json.load(fh)


def model_spec_dicts() -> list[dict]:
    return [asdict(m) for m in MODELS]


In [ ]:
%%writefile src/fetch_data.py
"""
Stage 01 - Build the dataset.

Downloads reviewed (Swiss-Prot) human proteins from the UniProt REST API in two
groups:

  Positives (label 1): annotated with the Transmembrane keyword (KW-0812).
  Negatives (label 0): annotated as located in the Cytoplasm (SL-0091) and
                       carrying neither the Membrane (KW-0472) nor the
                       Transmembrane (KW-0812) keyword.

The raw sets are then filtered by length, stripped of non-standard residues,
de-duplicated exactly, reduced for redundancy with a k-mer Jaccard filter, class
balanced, and split 70/15/15 in a stratified way.

Usage:
    python src/fetch_data.py --per-class 1500
    python src/fetch_data.py --mock            # offline synthetic data
"""
from __future__ import annotations

import argparse
import io
import re
import sys
import time

import numpy as np
import pandas as pd

sys.path.insert(0, str(__import__("pathlib").Path(__file__).resolve().parent))
from common import (AA, DATA, JACCARD_THRESHOLD, KMER_K, MAX_LEN, MIN_LEN,
                    SEED, SPLIT_FRACTIONS, SPLITS, get_logger, save_json,
                    set_seed)

log = get_logger("fetch")

UNIPROT_SEARCH = "https://rest.uniprot.org/uniprotkb/search"
FIELDS = "accession,id,protein_name,length,sequence"

QUERY_POS = (
    "(reviewed:true) AND (organism_id:9606) AND (keyword:KW-0812) "
    f"AND (length:[{MIN_LEN} TO {MAX_LEN}])"
)
QUERY_NEG = (
    "(reviewed:true) AND (organism_id:9606) AND (cc_scl_term:SL-0091) "
    "NOT (keyword:KW-0472) NOT (keyword:KW-0812) "
    f"AND (length:[{MIN_LEN} TO {MAX_LEN}])"
)

_NEXT_LINK = re.compile(r'<(?P<url>[^>]+)>;\s*rel="next"')


# --------------------------------------------------------------------------- #
# UniProt download
# --------------------------------------------------------------------------- #
def _uniprot_paginated(query: str, page_size: int = 500, max_records: int | None = None
                       ) -> pd.DataFrame:
    """Follow UniProt's Link-header pagination and concatenate TSV pages."""
    import requests

    session = requests.Session()
    session.headers.update({"User-Agent": "cse763-term-project/1.0"})

    url = UNIPROT_SEARCH
    params = {"query": query, "format": "tsv", "fields": FIELDS, "size": page_size}
    frames: list[pd.DataFrame] = []
    n = 0

    while url:
        for attempt in range(4):
            try:
                resp = session.get(url, params=params, timeout=90)
                resp.raise_for_status()
                break
            except Exception as exc:  # noqa: BLE001
                wait = 2 ** attempt
                log.warning("request failed (%s); retrying in %ss", exc, wait)
                time.sleep(wait)
        else:
            raise RuntimeError("UniProt request failed after 4 attempts")

        params = None  # the "next" URL already carries the query string
        page = pd.read_csv(io.StringIO(resp.text), sep="\t")
        if page.empty:
            break
        frames.append(page)
        n += len(page)
        log.info("  fetched %d records", n)

        if max_records is not None and n >= max_records:
            break

        m = _NEXT_LINK.search(resp.headers.get("Link", ""))
        url = m.group("url") if m else None

    if not frames:
        return pd.DataFrame(columns=FIELDS.split(","))
    return pd.concat(frames, ignore_index=True)


# --------------------------------------------------------------------------- #
# Mock data (offline development / CI)
# --------------------------------------------------------------------------- #
def _mock_records(n_per_class: int, rng: np.random.Generator) -> pd.DataFrame:
    """Synthetic sequences whose amino-acid composition differs by class.

    Transmembrane-like sequences are enriched in hydrophobic residues and carry
    short hydrophobic stretches; soluble-like sequences are enriched in charged
    and polar residues. This makes the downstream task genuinely learnable so the
    whole pipeline can be exercised without network access.
    """
    hydrophobic = "AILMFVWY"
    polar_charged = "DEKRNQSTHG"

    def draw(bias: str, strength: float, length: int) -> str:
        p = np.ones(len(AA))
        for ch in bias:
            p[AA.index(ch)] += strength
        p /= p.sum()
        return "".join(rng.choice(list(AA), size=length, p=p))

    rows = []
    for label in (1, 0):
        bias = hydrophobic if label == 1 else polar_charged
        for i in range(n_per_class):
            length = int(rng.integers(MIN_LEN, MAX_LEN))
            seq = draw(bias, strength=0.09, length=length)
            if label == 1:  # insert 1-3 hydrophobic "TM helices"
                for _ in range(int(rng.integers(0, 2))):
                    start = int(rng.integers(0, max(1, length - 15)))
                    helix = "".join(rng.choice(list(hydrophobic), size=14))
                    seq = seq[:start] + helix + seq[start + 14:]
            rows.append({
                "Entry": f"MOCK{label}{i:05d}",
                "Entry Name": f"MOCK{label}{i:05d}_HUMAN",
                "Protein names": "mock protein",
                "Length": len(seq),
                "Sequence": seq,
                "label": label,
            })
    return pd.DataFrame(rows)


# --------------------------------------------------------------------------- #
# Cleaning
# --------------------------------------------------------------------------- #
def _clean(df: pd.DataFrame, label: int) -> pd.DataFrame:
    df = df.rename(columns={c: c.strip() for c in df.columns})
    seq_col = "Sequence" if "Sequence" in df.columns else df.columns[-1]
    acc_col = "Entry" if "Entry" in df.columns else df.columns[0]

    out = pd.DataFrame({
        "accession": df[acc_col].astype(str),
        "sequence": df[seq_col].astype(str).str.upper().str.strip(),
    })
    out["label"] = label
    out = out[out.sequence.str.len().between(MIN_LEN, MAX_LEN)]
    # keep only the 20 standard amino acids (drops X, B, Z, U, O)
    valid = set(AA)
    out = out[out.sequence.map(lambda s: set(s) <= valid)]
    out = out.drop_duplicates(subset="sequence")
    return out.reset_index(drop=True)


def _kmer_matrix(seqs: list[str], k: int = KMER_K):
    """Boolean sparse matrix of k-mer presence, shape (n_seqs, n_distinct_kmers)."""
    from scipy.sparse import csr_matrix

    vocab: dict[str, int] = {}
    indptr, indices = [0], []
    for s in seqs:
        kmers = {s[i:i + k] for i in range(len(s) - k + 1)}
        for km in kmers:
            indices.append(vocab.setdefault(km, len(vocab)))
        indptr.append(len(indices))
    data = np.ones(len(indices), dtype=np.float32)
    return csr_matrix((data, indices, indptr), shape=(len(seqs), len(vocab)))


def reduce_redundancy(df: pd.DataFrame, threshold: float = JACCARD_THRESHOLD
                      ) -> pd.DataFrame:
    """Greedy removal of near-duplicate sequences by k-mer Jaccard similarity.

    Cheap stand-in for CD-HIT clustering. Applied across BOTH classes before
    splitting, so near-identical proteins cannot leak between train and test.
    """
    seqs = df.sequence.tolist()
    if len(seqs) < 2:
        return df

    X = _kmer_matrix(seqs)
    sizes = np.asarray(X.sum(axis=1)).ravel()
    inter = (X @ X.T).toarray()                       # |A ∩ B|
    union = sizes[:, None] + sizes[None, :] - inter   # |A ∪ B|
    jac = np.divide(inter, np.maximum(union, 1e-9))
    np.fill_diagonal(jac, 0.0)

    order = np.argsort(-np.array([len(s) for s in seqs]))  # prefer longer seqs
    kept: list[int] = []
    for idx in order:
        if not kept or jac[idx, kept].max() < threshold:
            kept.append(int(idx))

    log.info("redundancy filter: %d -> %d sequences (Jaccard >= %.2f removed)",
             len(seqs), len(kept), threshold)
    return df.iloc[sorted(kept)].reset_index(drop=True)


def stratified_split(df: pd.DataFrame, rng: np.random.Generator) -> pd.DataFrame:
    df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    df["split"] = "train"
    tr, va, _ = SPLIT_FRACTIONS
    for label, grp in df.groupby("label"):
        idx = grp.index.to_numpy()
        n = len(idx)
        n_tr, n_va = int(round(tr * n)), int(round(va * n))
        df.loc[idx[n_tr:n_tr + n_va], "split"] = "val"
        df.loc[idx[n_tr + n_va:], "split"] = "test"
    return df


# --------------------------------------------------------------------------- #
def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--per-class", type=int, default=1500,
                    help="target sequences per class after all filtering")
    ap.add_argument("--mock", action="store_true",
                    help="generate synthetic data instead of querying UniProt")
    ap.add_argument("--no-redundancy-filter", action="store_true")
    args = ap.parse_args()

    set_seed()
    rng = np.random.default_rng(SEED)

    if args.mock:
        log.info("MOCK MODE - generating synthetic sequences")
        raw = _mock_records(int(args.per_class * 1.4), rng)
        pos = _clean(raw[raw.label == 1], 1)
        neg = _clean(raw[raw.label == 0], 0)
    else:
        # over-fetch, because filtering discards a sizeable fraction
        budget = int(args.per_class * 3)
        log.info("querying UniProt for transmembrane proteins ...")
        pos = _clean(_uniprot_paginated(QUERY_POS, max_records=budget), 1)
        log.info("querying UniProt for soluble cytoplasmic proteins ...")
        neg = _clean(_uniprot_paginated(QUERY_NEG, max_records=budget), 0)

    log.info("after cleaning: %d positive, %d negative", len(pos), len(neg))
    if min(len(pos), len(neg)) < 100:
        raise SystemExit("Too few sequences retrieved - check the UniProt queries.")

    df = pd.concat([pos, neg], ignore_index=True)
    if not args.no_redundancy_filter:
        df = reduce_redundancy(df)

    # balance the classes, then trim to the requested size
    n = int(min(args.per_class, df.label.value_counts().min()))
    df = pd.concat(
        [df[df.label == lab].sample(n=n, random_state=SEED) for lab in (0, 1)],
        ignore_index=True,
    )
    log.info("balanced dataset: %d per class (%d total)", n, len(df))

    df = stratified_split(df, rng)
    for split in ("train", "val", "test"):
        sub = df[df.split == split].drop(columns="split").reset_index(drop=True)
        sub.to_csv(SPLITS / f"{split}.csv", index=False)
        log.info("%-5s -> %4d sequences (%d positive)", split, len(sub), int(sub.label.sum()))

    summary = {
        "mock": bool(args.mock),
        "per_class": int(n),
        "total": int(len(df)),
        "redundancy_filter": not args.no_redundancy_filter,
        "jaccard_threshold": JACCARD_THRESHOLD,
        "kmer_k": KMER_K,
        "length_window": [MIN_LEN, MAX_LEN],
        "query_positive": QUERY_POS,
        "query_negative": QUERY_NEG,
        "splits": {s: int((df.split == s).sum()) for s in ("train", "val", "test")},
        "mean_length": float(df.sequence.str.len().mean()),
        "median_length": float(df.sequence.str.len().median()),
    }
    save_json(summary, DATA / "dataset_summary.json")
    log.info("wrote %s", DATA / "dataset_summary.json")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/embed.py
"""
Stage 02 - Extract frozen embeddings.

For each checkpoint, every sequence passes once through the frozen encoder and
is reduced to a fixed-length vector. Four pooling/layer variants are stored from
the same forward pass so that the ablation study (Stage 06) costs no extra GPU
time:

    mean_last  mean over residue positions, final layer      (primary)
    cls_last   the <cls> token representation, final layer
    max_last   element-wise max over residue positions, final layer
    mean_mid   mean over residue positions, middle layer

Wall-clock time and peak GPU memory are recorded per model.

Usage:
    python src/embed.py
    python src/embed.py --models esm2_8m --overwrite
    python src/embed.py --mock
"""
from __future__ import annotations

import argparse
import os
import sys
import time
from pathlib import Path

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parent))
from common import (EMB, MODELS, MODELS_BY_KEY, SEED, SPLITS, VARIANTS,
                    ModelSpec, get_logger, save_json, set_seed)

log = get_logger("embed")


def load_split(split: str) -> pd.DataFrame:
    path = SPLITS / f"{split}.csv"
    if not path.exists():
        raise SystemExit(f"{path} missing - run src/fetch_data.py first.")
    return pd.read_csv(path)


# --------------------------------------------------------------------------- #
# Mock embeddings
# --------------------------------------------------------------------------- #
def mock_embed(spec: ModelSpec, seqs: list[str]) -> dict[str, np.ndarray]:
    """Synthetic embeddings for offline pipeline testing.

    Built from real amino-acid composition plus noise whose scale shrinks with
    model size. Variants are given deliberately different noise levels so the
    ablation machinery can be exercised. These are NOT protein representations
    and must never be reported as results.
    """
    import zlib

    from common import AA
    rng = np.random.default_rng(SEED + zlib.crc32(spec.key.encode()) % 10_000)
    n, d = len(seqs), spec.dim

    idx = {a: i for i, a in enumerate(AA)}
    F = np.zeros((n, len(AA) + 1), dtype=np.float32)
    for r, s in enumerate(seqs):
        for ch in s:
            j = idx.get(ch)
            if j is not None:
                F[r, j] += 1
        F[r, :len(AA)] /= max(len(s), 1)
        F[r, -1] = np.log(len(s))
    F = (F - F.mean(0)) / (F.std(0) + 1e-8)

    base = 8.0 if spec.random_init else \
        {8: 2.6, 35: 2.1, 150: 1.7, 650: 1.5}[spec.params // 1_000_000]
    penalty = {"mean_last": 1.0, "cls_last": 1.35, "max_last": 1.15, "mean_mid": 1.05}

    out = {}
    for v in VARIANTS:
        W = rng.normal(size=(F.shape[1], d)) / np.sqrt(F.shape[1])
        noise = base * penalty[v]
        out[v] = (F @ W + noise * rng.normal(size=(n, d))).astype(np.float32)
    return out


# --------------------------------------------------------------------------- #
# Real embeddings
# --------------------------------------------------------------------------- #
def _batched_by_length(seqs: list[str], max_tokens: int):
    """Yield (indices, sequences) batches of similar length to limit padding waste."""
    order = sorted(range(len(seqs)), key=lambda i: len(seqs[i]))
    batch, batch_max = [], 0
    for i in order:
        L = len(seqs[i]) + 2
        new_max = max(batch_max, L)
        if batch and new_max * (len(batch) + 1) > max_tokens:
            yield batch, [seqs[j] for j in batch]
            batch, batch_max = [i], L
        else:
            batch.append(i)
            batch_max = new_max
    if batch:
        yield batch, [seqs[j] for j in batch]


def _pool(hidden_last, hidden_mid, attn_mask, device):
    """Return the four pooled variants for one batch."""
    import torch

    # Residue mask: drop padding, <cls> (position 0) and <eos> (last real token).
    mask = attn_mask.clone()
    mask[:, 0] = 0
    last_idx = attn_mask.sum(1) - 1
    mask[torch.arange(mask.size(0), device=device), last_idx] = 0
    m = mask.unsqueeze(-1).to(hidden_last.dtype)
    denom = m.sum(1).clamp(min=1)

    out = {
        "mean_last": (hidden_last * m).sum(1) / denom,
        "cls_last": hidden_last[:, 0, :],
        "max_last": (hidden_last.masked_fill(m == 0, -1e4)).max(dim=1).values,
        "mean_mid": (hidden_mid * m).sum(1) / denom,
    }
    return {k: v.float() for k, v in out.items()}


def real_embed(spec: ModelSpec, seqs: list[str], max_tokens: int, fp16: bool
               ) -> tuple[dict[str, np.ndarray], dict]:
    import torch
    from transformers import AutoConfig, AutoTokenizer, EsmModel

    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if (fp16 and device == "cuda") else torch.float32

    tok = AutoTokenizer.from_pretrained(spec.hf_id)
    if spec.random_init:
        cfg = AutoConfig.from_pretrained(spec.hf_id)
        torch.manual_seed(SEED)
        model = EsmModel(cfg, add_pooling_layer=False)
        log.info("  [control] randomly initialised weights, no pretraining")
    else:
        model = EsmModel.from_pretrained(spec.hf_id, add_pooling_layer=False)
    model = model.to(device=device, dtype=dtype).eval()

    n_params = sum(p.numel() for p in model.parameters())
    n_layers = model.config.num_hidden_layers
    mid_layer = n_layers // 2
    log.info("  %s on %s | %.1fM parameters | %d layers (mid=%d) | dtype=%s",
             spec.hf_id, device, n_params / 1e6, n_layers, mid_layer, dtype)

    D = model.config.hidden_size
    out = {v: np.zeros((len(seqs), D), dtype=np.float32) for v in VARIANTS}

    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    done = 0

    with torch.no_grad():
        for idx, batch in _batched_by_length(seqs, max_tokens):
            enc = tok(batch, return_tensors="pt", padding=True,
                      truncation=True, max_length=1024)
            enc = {k: v.to(device) for k, v in enc.items()}
            res = model(**enc, output_hidden_states=True)
            hs = res.hidden_states
            pooled = _pool(hs[-1], hs[mid_layer], enc["attention_mask"], device)

            if any(torch.isnan(v).any() for v in pooled.values()):
                log.warning("    NaN in fp16 output; recomputing batch in fp32")
                model.float()
                res32 = model(**enc, output_hidden_states=True)
                hs32 = res32.hidden_states
                pooled = _pool(hs32[-1], hs32[mid_layer], enc["attention_mask"], device)
                model.to(dtype)

            for v in VARIANTS:
                out[v][idx] = pooled[v].cpu().numpy()

            done += len(batch)
            if done % 500 < len(batch):
                log.info("    %d / %d sequences", done, len(seqs))

    elapsed = time.time() - t0
    peak_mem = (torch.cuda.max_memory_allocated() / 1e9) if device == "cuda" else 0.0

    del model
    if device == "cuda":
        torch.cuda.empty_cache()

    return out, {"seconds": round(elapsed, 2), "peak_gpu_gb": round(peak_mem, 3),
                 "n_params": int(n_params), "n_layers": n_layers,
                 "mid_layer": mid_layer, "device": device, "dtype": str(dtype)}


# --------------------------------------------------------------------------- #
def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--models", nargs="*", default=None)
    ap.add_argument("--mock", action="store_true")
    ap.add_argument("--max-tokens", type=int, default=8192,
                    help="tokens per batch; lower this if you hit OOM")
    ap.add_argument("--fp16", action="store_true", default=True)
    ap.add_argument("--fp32", dest="fp16", action="store_false")
    ap.add_argument("--overwrite", action="store_true")
    args = ap.parse_args()

    set_seed()
    specs = ([MODELS_BY_KEY[k] for k in args.models] if args.models else MODELS)
    splits = {s: load_split(s) for s in ("train", "val", "test")}

    timings: dict[str, dict] = {}
    timings_path = EMB / "timings.json"
    if timings_path.exists():
        import json
        timings = json.loads(timings_path.read_text())

    for spec in specs:
        if (EMB / f"{spec.key}_test.npy").exists() and not args.overwrite:
            log.info("%s already embedded - skipping (use --overwrite to redo)", spec.key)
            continue

        log.info("=== %s ===", spec.label)
        all_seqs, bounds, cursor = [], {}, 0
        for name, df in splits.items():
            bounds[name] = (cursor, cursor + len(df))
            cursor += len(df)
            all_seqs.extend(df.sequence.tolist())

        if args.mock:
            t0 = time.time()
            mats = mock_embed(spec, all_seqs)
            meta = {"seconds": round(time.time() - t0, 2), "peak_gpu_gb": 0.0,
                    "n_params": spec.params, "n_layers": spec.layers,
                    "mid_layer": spec.layers // 2, "device": "mock", "dtype": "mock"}
        else:
            mats, meta = real_embed(spec, all_seqs, args.max_tokens, args.fp16)

        for name, (a, b) in bounds.items():
            # primary variant keeps the plain filename
            np.save(EMB / f"{spec.key}_{name}.npy", mats["mean_last"][a:b])
            for v in VARIANTS:
                np.save(EMB / f"{spec.key}_{v}_{name}.npy", mats[v][a:b])

        timings[spec.key] = meta
        save_json(timings, timings_path)
        log.info("  done in %.1fs | peak GPU %.2f GB | %d variants saved",
                 meta["seconds"], meta["peak_gpu_gb"], len(VARIANTS))

    log.info("embeddings written to %s", EMB)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/probe.py
"""
Stage 03 - Linear probes and baselines.

Protocol, identical for every representation:
  1. Standardise features using statistics from the training split only.
  2. Sweep the L2 regularisation strength C of a logistic regression on train,
     selecting the value with the best validation macro-F1.
  3. Refit on train + validation with the chosen C.
  4. Report accuracy, macro-F1, MCC and AUROC on the held-out test split, with
     95% bootstrap confidence intervals (2000 resamples).

Because the encoders are frozen, differences between rows are attributable to
the representations rather than to classifier capacity.

Usage:
    python src/probe.py
"""
from __future__ import annotations

import sys
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, matthews_corrcoef,
                             roc_auc_score)
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, str(Path(__file__).resolve().parent))
from common import (AA, EMB, MODELS, RESULTS, SEED, SPLITS, get_logger,
                    load_json, save_json, set_seed)

log = get_logger("probe")

C_GRID = [1e-3, 1e-2, 1e-1, 1.0, 10.0]
SWEEP_ROWS: list[dict] = []
N_BOOTSTRAP = 2000


# --------------------------------------------------------------------------- #
# Baseline featurisers
# --------------------------------------------------------------------------- #
def aac_features(seqs: list[str]) -> np.ndarray:
    """20 amino-acid frequencies + log length."""
    idx = {a: i for i, a in enumerate(AA)}
    X = np.zeros((len(seqs), len(AA) + 1), dtype=np.float32)
    for r, s in enumerate(seqs):
        for ch in s:
            j = idx.get(ch)
            if j is not None:
                X[r, j] += 1
        X[r, :len(AA)] /= max(len(s), 1)
        X[r, -1] = np.log(len(s))
    return X


def kmer_features(seqs: list[str], k: int = 3) -> np.ndarray:
    """Normalised k-mer frequency vector over the 20-letter alphabet."""
    vocab = {"".join(p): i for i, p in enumerate(product(AA, repeat=k))}
    X = np.zeros((len(seqs), len(vocab)), dtype=np.float32)
    for r, s in enumerate(seqs):
        for i in range(len(s) - k + 1):
            j = vocab.get(s[i:i + k])
            if j is not None:
                X[r, j] += 1
        total = X[r].sum()
        if total:
            X[r] /= total
    return X


# --------------------------------------------------------------------------- #
# Evaluation
# --------------------------------------------------------------------------- #
def bootstrap_ci(y_true: np.ndarray, y_pred: np.ndarray, y_score: np.ndarray,
                 fn, n: int = N_BOOTSTRAP) -> tuple[float, float]:
    rng = np.random.default_rng(SEED)
    vals = []
    m = len(y_true)
    for _ in range(n):
        idx = rng.integers(0, m, m)
        if len(np.unique(y_true[idx])) < 2:
            continue
        vals.append(fn(y_true[idx], y_pred[idx], y_score[idx]))
    if not vals:
        return (float("nan"), float("nan"))
    return (float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5)))


METRICS = {
    "accuracy": lambda yt, yp, ys: accuracy_score(yt, yp),
    "macro_f1": lambda yt, yp, ys: f1_score(yt, yp, average="macro"),
    "mcc":      lambda yt, yp, ys: matthews_corrcoef(yt, yp),
    "auroc":    lambda yt, yp, ys: roc_auc_score(yt, ys),
}


def run_probe(name: str, Xtr, ytr, Xva, yva, Xte, yte) -> dict:
    scaler = StandardScaler().fit(Xtr)
    Xtr_s, Xva_s, Xte_s = scaler.transform(Xtr), scaler.transform(Xva), scaler.transform(Xte)

    best_c, best_f1 = None, -1.0
    sweep = []
    for C in C_GRID:
        clf = LogisticRegression(C=C, max_iter=3000, random_state=SEED)
        clf.fit(Xtr_s, ytr)
        f1 = f1_score(yva, clf.predict(Xva_s), average="macro")
        auc = roc_auc_score(yva, clf.predict_proba(Xva_s)[:, 1])
        sweep.append({"representation": name, "C": C,
                      "val_macro_f1": round(float(f1), 4),
                      "val_auroc": round(float(auc), 4)})
        log.info("    C=%-6g  val macro-F1=%.4f  val AUROC=%.4f", C, f1, auc)
        if f1 > best_f1:
            best_c, best_f1 = C, f1
    SWEEP_ROWS.extend(sweep)

    # refit on train + val with the selected C
    Xfull = np.vstack([Xtr, Xva])
    yfull = np.concatenate([ytr, yva])
    scaler = StandardScaler().fit(Xfull)
    clf = LogisticRegression(C=best_c, max_iter=3000, random_state=SEED)
    clf.fit(scaler.transform(Xfull), yfull)

    Xte_s = scaler.transform(Xte)
    y_pred = clf.predict(Xte_s)
    y_score = clf.predict_proba(Xte_s)[:, 1]

    row = {"representation": name, "dim": int(Xtr.shape[1]),
           "best_C": best_c, "val_macro_f1": round(best_f1, 4)}
    for mname, fn in METRICS.items():
        row[mname] = round(float(fn(yte, y_pred, y_score)), 4)
        lo, hi = bootstrap_ci(yte, y_pred, y_score, fn)
        row[f"{mname}_lo"], row[f"{mname}_hi"] = round(lo, 4), round(hi, 4)

    np.save(RESULTS / f"scores_{name.replace('/', '_').replace(' ', '_')}.npy",
            np.vstack([yte, y_score]))
    log.info("  %-28s acc=%.4f  F1=%.4f  MCC=%.4f  AUROC=%.4f",
             name, row["accuracy"], row["macro_f1"], row["mcc"], row["auroc"])
    return row


# --------------------------------------------------------------------------- #
def main() -> None:
    set_seed()
    splits = {s: pd.read_csv(SPLITS / f"{s}.csv") for s in ("train", "val", "test")}
    y = {s: df.label.to_numpy() for s, df in splits.items()}
    seqs = {s: df.sequence.tolist() for s, df in splits.items()}

    rows: list[dict] = []

    log.info("=== baseline: amino-acid composition + length ===")
    feats = {s: aac_features(seqs[s]) for s in splits}
    rows.append(run_probe("Baseline: AAC + length",
                          feats["train"], y["train"], feats["val"], y["val"],
                          feats["test"], y["test"]))

    log.info("=== baseline: 3-mer frequencies ===")
    feats = {s: kmer_features(seqs[s], k=3) for s in splits}
    rows.append(run_probe("Baseline: 3-mer frequency",
                          feats["train"], y["train"], feats["val"], y["val"],
                          feats["test"], y["test"]))

    timings = load_json(EMB / "timings.json") if (EMB / "timings.json").exists() else {}

    for spec in MODELS:
        path = EMB / f"{spec.key}_test.npy"
        if not path.exists():
            log.warning("no embeddings for %s - skipping", spec.key)
            continue
        log.info("=== %s ===", spec.label)
        X = {s: np.load(EMB / f"{spec.key}_{s}.npy") for s in splits}
        row = run_probe(spec.label, X["train"], y["train"], X["val"], y["val"],
                        X["test"], y["test"])
        row["model_key"] = spec.key
        row["params"] = spec.params
        row["layers"] = spec.layers
        row["pretrained"] = not spec.random_init
        t = timings.get(spec.key, {})
        row["embed_seconds"] = t.get("seconds")
        row["peak_gpu_gb"] = t.get("peak_gpu_gb")
        rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv(RESULTS / "results.csv", index=False)

    # human-readable table for the report
    cols = ["representation", "dim", "accuracy", "macro_f1", "mcc", "auroc"]
    md = df[cols].copy()
    for m in ("accuracy", "macro_f1", "mcc", "auroc"):
        md[m] = df.apply(
            lambda r, m=m: f"{r[m]:.3f} [{r[f'{m}_lo']:.3f}, {r[f'{m}_hi']:.3f}]", axis=1)
    (RESULTS / "results.md").write_text(
        "# Test-set results (95% bootstrap CI)\n\n" +
        md.to_markdown(index=False) + "\n")

    sweep_df = pd.DataFrame(SWEEP_ROWS)
    sweep_df.to_csv(RESULTS / "hyperparameter_sweep.csv", index=False)
    piv = sweep_df.pivot_table(index="representation", columns="C",
                               values="val_macro_f1")
    (RESULTS / "hyperparameter_sweep.md").write_text(
        "# Validation macro-F1 across the regularisation grid\n\n"
        + piv.round(4).to_markdown() + "\n")

    save_json({"n_test": int(len(y["test"])), "n_bootstrap": N_BOOTSTRAP,
               "C_grid": C_GRID}, RESULTS / "probe_config.json")
    log.info("wrote %s and %s", RESULTS / "results.csv", RESULTS / "results.md")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/ablation.py
"""
Stage 06 - Ablation study, learning curves and convergence curves.

Three experiments, all required by the report:

  A. POOLING / LAYER ABLATION
     The primary pipeline uses mean pooling over the final layer. Here we swap
     that component for three alternatives (<cls> token, max pooling, middle
     layer) while holding everything else fixed, isolating the contribution of
     the pooling and layer choice.

  B. PREPROCESSING / PROBE ABLATION
     Removes one pipeline component at a time: feature standardisation,
     regularisation tuning, and the redundancy filter's effect via a
     shuffled-label control.

  C. LEARNING AND CONVERGENCE CURVES
     Learning curves show test performance against training-set size.
     Convergence curves show train and validation log-loss per epoch for an
     SGD-trained logistic regression, which the closed-form solver does not
     expose.

Usage:
    python src/ablation.py
"""
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import f1_score, log_loss, roc_auc_score
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, str(Path(__file__).resolve().parent))
from common import (EMB, MODELS, RESULTS, SEED, SPLITS, VARIANT_LABELS,
                    VARIANTS, get_logger, save_json, set_seed)

log = get_logger("ablation")

TRAIN_FRACTIONS = [0.05, 0.10, 0.25, 0.50, 0.75, 1.0]
N_EPOCHS = 60


def load_labels():
    return {s: pd.read_csv(SPLITS / f"{s}.csv").label.to_numpy()
            for s in ("train", "val", "test")}


def fit_eval(Xtr, ytr, Xte, yte, C=1.0, standardise=True):
    if standardise:
        sc = StandardScaler().fit(Xtr)
        Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)
    clf = LogisticRegression(C=C, max_iter=3000, random_state=SEED).fit(Xtr, ytr)
    pred = clf.predict(Xte)
    score = clf.predict_proba(Xte)[:, 1]
    return {"macro_f1": f1_score(yte, pred, average="macro"),
            "auroc": roc_auc_score(yte, score)}


# --------------------------------------------------------------------------- #
# A. Pooling / layer ablation
# --------------------------------------------------------------------------- #
def ablation_pooling(y) -> pd.DataFrame:
    rows = []
    for spec in MODELS:
        if not (EMB / f"{spec.key}_test.npy").exists():
            continue
        for v in VARIANTS:
            paths = {s: EMB / f"{spec.key}_{v}_{s}.npy" for s in ("train", "test")}
            if not all(p.exists() for p in paths.values()):
                log.warning("missing variant %s for %s - skipping", v, spec.key)
                continue
            X = {s: np.load(p) for s, p in paths.items()}
            m = fit_eval(X["train"], y["train"], X["test"], y["test"])
            rows.append({"model": spec.label, "params": spec.params,
                         "variant": v, "variant_label": VARIANT_LABELS[v],
                         "pretrained": not spec.random_init, **m})
            log.info("  %-24s %-24s F1=%.4f AUROC=%.4f",
                     spec.label, v, m["macro_f1"], m["auroc"])
    return pd.DataFrame(rows)


# --------------------------------------------------------------------------- #
# B. Preprocessing / probe ablation
# --------------------------------------------------------------------------- #
def ablation_components(y) -> pd.DataFrame:
    """Remove one component at a time from the best pretrained model."""
    pre = [s for s in MODELS if not s.random_init
           and (EMB / f"{s.key}_test.npy").exists()]
    if not pre:
        return pd.DataFrame()
    spec = max(pre, key=lambda s: s.params)
    X = {s: np.load(EMB / f"{spec.key}_{s}.npy") for s in ("train", "val", "test")}
    log.info("component ablation on %s", spec.label)

    rows = []

    full = fit_eval(X["train"], y["train"], X["test"], y["test"], C=1.0)
    rows.append({"setting": "Full pipeline", **full})

    nostd = fit_eval(X["train"], y["train"], X["test"], y["test"],
                     C=1.0, standardise=False)
    rows.append({"setting": "No feature standardisation", **nostd})

    weak = fit_eval(X["train"], y["train"], X["test"], y["test"], C=1e-3)
    rows.append({"setting": "Heavy regularisation (C=0.001)", **weak})

    strong = fit_eval(X["train"], y["train"], X["test"], y["test"], C=10.0)
    rows.append({"setting": "Weak regularisation (C=10)", **strong})

    # Shuffled-label control: confirms the pipeline reports chance when the
    # relationship between features and labels is destroyed.
    rng = np.random.default_rng(SEED)
    y_shuf = rng.permutation(y["train"])
    shuf = fit_eval(X["train"], y_shuf, X["test"], y["test"], C=1.0)
    rows.append({"setting": "Shuffled training labels (sanity check)", **shuf})

    # Dimensionality: truncate the embedding to its first 64 components.
    trunc = fit_eval(X["train"][:, :64], y["train"], X["test"][:, :64], y["test"])
    rows.append({"setting": "First 64 dimensions only", **trunc})

    df = pd.DataFrame(rows)
    df["model"] = spec.label
    for _, r in df.iterrows():
        log.info("  %-42s F1=%.4f AUROC=%.4f", r.setting, r.macro_f1, r.auroc)
    return df


# --------------------------------------------------------------------------- #
# C. Learning curves and convergence curves
# --------------------------------------------------------------------------- #
def learning_curves(y) -> pd.DataFrame:
    rows = []
    rng = np.random.default_rng(SEED)
    for spec in MODELS:
        if not (EMB / f"{spec.key}_test.npy").exists():
            continue
        Xtr = np.load(EMB / f"{spec.key}_train.npy")
        Xte = np.load(EMB / f"{spec.key}_test.npy")
        n = len(Xtr)
        for frac in TRAIN_FRACTIONS:
            k = max(int(frac * n), 20)
            idx = rng.choice(n, k, replace=False)
            if len(np.unique(y["train"][idx])) < 2:
                continue
            m = fit_eval(Xtr[idx], y["train"][idx], Xte, y["test"])
            rows.append({"model": spec.label, "params": spec.params,
                         "pretrained": not spec.random_init,
                         "frac": frac, "n_train": k, **m})
        log.info("  learning curve done for %s", spec.label)
    return pd.DataFrame(rows)


def convergence_curves(y) -> pd.DataFrame:
    """Per-epoch train and validation log-loss for an SGD-trained probe."""
    rows = []
    for spec in MODELS:
        if not (EMB / f"{spec.key}_test.npy").exists():
            continue
        Xtr = np.load(EMB / f"{spec.key}_train.npy")
        Xva = np.load(EMB / f"{spec.key}_val.npy")
        sc = StandardScaler().fit(Xtr)
        Xtr_s, Xva_s = sc.transform(Xtr), sc.transform(Xva)

        clf = SGDClassifier(loss="log_loss", alpha=1e-2, learning_rate="optimal",
                            average=True, random_state=SEED)
        classes = np.array([0, 1])
        rng = np.random.default_rng(SEED)
        for epoch in range(1, N_EPOCHS + 1):
            order = rng.permutation(len(Xtr_s))
            clf.partial_fit(Xtr_s[order], y["train"][order], classes=classes)
            ptr = np.clip(clf.predict_proba(Xtr_s)[:, 1], 1e-6, 1 - 1e-6)
            pva = np.clip(clf.predict_proba(Xva_s)[:, 1], 1e-6, 1 - 1e-6)
            rows.append({
                "model": spec.label, "params": spec.params,
                "pretrained": not spec.random_init, "epoch": epoch,
                "train_loss": log_loss(y["train"], ptr, labels=[0, 1]),
                "val_loss": log_loss(y["val"], pva, labels=[0, 1]),
                "val_auroc": roc_auc_score(y["val"], pva),
            })
        log.info("  convergence curve done for %s", spec.label)
    return pd.DataFrame(rows)


# --------------------------------------------------------------------------- #
def main() -> None:
    set_seed()
    y = load_labels()

    log.info("=== A. pooling / layer ablation ===")
    pooling = ablation_pooling(y)
    pooling.to_csv(RESULTS / "ablation_pooling.csv", index=False)

    log.info("=== B. component ablation ===")
    comp = ablation_components(y)
    comp.to_csv(RESULTS / "ablation_components.csv", index=False)

    log.info("=== C. learning curves ===")
    lc = learning_curves(y)
    lc.to_csv(RESULTS / "learning_curves.csv", index=False)

    log.info("=== C. convergence curves ===")
    cc = convergence_curves(y)
    cc.to_csv(RESULTS / "convergence_curves.csv", index=False)

    # Markdown tables for direct use in the report
    lines = ["# Ablation study\n", "\n## A. Pooling and layer choice\n"]
    if not pooling.empty:
        piv = pooling.pivot_table(index="model", columns="variant_label",
                                  values="auroc")
        lines.append(piv.round(4).to_markdown() + "\n")
    lines.append("\n## B. Pipeline components\n")
    if not comp.empty:
        lines.append(comp[["setting", "macro_f1", "auroc"]].round(4)
                     .to_markdown(index=False) + "\n")
    (RESULTS / "ablation.md").write_text("\n".join(lines))

    save_json({"train_fractions": TRAIN_FRACTIONS, "n_epochs": N_EPOCHS,
               "variants": VARIANTS}, RESULTS / "ablation_config.json")
    log.info("wrote ablation results to %s", RESULTS)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/figures.py
"""
Stage 04 - Figures.

  fig1_scaling.png        AUROC and macro-F1 against model size (log axis),
                          with baselines and the random-init control marked.
  fig2_embedding_space.png  2-D projection (UMAP if installed, otherwise PCA)
                          of test embeddings, coloured by class, smallest vs
                          largest pretrained model side by side.
  fig3_compute.png        AUROC against embedding wall-clock time.
  fig4_roc.png            ROC curves for every representation.

Usage:
    python src/figures.py
"""
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.metrics import roc_curve

sys.path.insert(0, str(Path(__file__).resolve().parent))
from common import (CLASS_NAMES, EMB, FIGURES, RESULTS, SEED, SPLITS,
                    get_logger, set_seed)

log = get_logger("figures")
plt.rcParams.update({"figure.dpi": 150, "font.size": 9,
                     "axes.spines.top": False, "axes.spines.right": False})

BLUE, ORANGE, GREY, RED = "#2b6cb0", "#dd6b20", "#718096", "#c53030"


def load_results() -> pd.DataFrame:
    path = RESULTS / "results.csv"
    if not path.exists():
        raise SystemExit("results.csv missing - run src/probe.py first.")
    return pd.read_csv(path)


# --------------------------------------------------------------------------- #
def fig_scaling(df: pd.DataFrame) -> None:
    is_base = df.representation.str.startswith("Baseline")
    pre = df[df.pretrained & ~is_base].sort_values("params")
    rnd = df[~df.pretrained & ~is_base]
    base = df[is_base]

    fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
    for ax, metric, title in zip(axes, ["auroc", "macro_f1"],
                                 ["AUROC", "Macro-F1"]):
        yerr = np.vstack([pre[metric] - pre[f"{metric}_lo"],
                          pre[f"{metric}_hi"] - pre[metric]])
        ax.errorbar(pre.params, pre[metric], yerr=yerr, marker="o", ms=6,
                    lw=1.8, capsize=3, color=BLUE, label="ESM-2 (pretrained)")
        for _, r in base.iterrows():
            ax.axhline(r[metric], ls="--", lw=1.2, color=GREY)
            ax.text(pre.params.max(), r[metric], f"{r.representation} ",
                    fontsize=7, color=GREY, va="bottom", ha="right")
        if len(rnd):
            r = rnd.iloc[0]
            ax.scatter([r.params], [r[metric]], marker="X", s=90, color=RED,
                       zorder=5, label="8M, random init (control)")
        ax.set_xscale("log")
        ax.set_xlabel("Encoder parameters")
        ax.set_ylabel(title)
        ax.set_title(f"{title} vs. model scale")
        ax.grid(alpha=0.25, ls=":")
        lo, hi = ax.get_ylim()
        ax.set_ylim(lo - 0.04 * (hi - lo), hi + 0.06 * (hi - lo))
    axes[0].legend(fontsize=7, loc="lower right", framealpha=0.95)
    fig.tight_layout()
    fig.savefig(FIGURES / "fig1_scaling.png", bbox_inches="tight")
    plt.close(fig)
    log.info("wrote fig1_scaling.png")


def fig_embedding_space(df: pd.DataFrame) -> None:
    pre = df[df.pretrained == True].sort_values("params")            # noqa: E712
    if pre.empty:
        return
    keys = [pre.iloc[0].model_key, pre.iloc[-1].model_key]
    labels = [pre.iloc[0].representation, pre.iloc[-1].representation]
    y = pd.read_csv(SPLITS / "test.csv").label.to_numpy()

    try:
        import umap
        reducer_name = "UMAP"
        make = lambda: umap.UMAP(n_neighbors=25, min_dist=0.1, random_state=SEED)
    except ImportError:
        reducer_name = "PCA"
        make = lambda: PCA(n_components=2, random_state=SEED)
        log.info("umap-learn not installed - falling back to PCA")

    fig, axes = plt.subplots(1, len(keys), figsize=(4.4 * len(keys), 4.0))
    axes = np.atleast_1d(axes)
    for ax, key, lab in zip(axes, keys, labels):
        X = np.load(EMB / f"{key}_test.npy")
        Z = make().fit_transform(X)
        for cls, colour in ((0, BLUE), (1, ORANGE)):
            m = y == cls
            ax.scatter(Z[m, 0], Z[m, 1], s=7, alpha=0.6, c=colour,
                       label=CLASS_NAMES[cls], linewidths=0)
        ax.set_title(f"{lab}\n({reducer_name} of test embeddings)")
        ax.set_xticks([]); ax.set_yticks([])
    axes[0].legend(fontsize=8, markerscale=2, loc="best")
    fig.tight_layout()
    fig.savefig(FIGURES / "fig2_embedding_space.png", bbox_inches="tight")
    plt.close(fig)
    log.info("wrote fig2_embedding_space.png")


def fig_compute(df: pd.DataFrame) -> None:
    sub = df[df.embed_seconds.notna() & (df.pretrained == True)]     # noqa: E712
    if sub.empty or sub.embed_seconds.max() <= 0:
        log.info("no timing data - skipping fig3")
        return
    fig, ax = plt.subplots(figsize=(4.8, 3.6))
    ax.scatter(sub.embed_seconds, sub.auroc, s=70, color=BLUE, zorder=3)
    for _, r in sub.iterrows():
        ax.annotate(r.representation, (r.embed_seconds, r.auroc),
                    textcoords="offset points", xytext=(6, -3), fontsize=7)
    ax.set_xlabel("Embedding wall-clock time for the full dataset (s)")
    ax.set_ylabel("Test AUROC")
    ax.set_title("Accuracy vs. compute cost")
    ax.grid(alpha=0.25, ls=":")
    fig.tight_layout()
    fig.savefig(FIGURES / "fig3_compute.png", bbox_inches="tight")
    plt.close(fig)
    log.info("wrote fig3_compute.png")


def fig_roc(df: pd.DataFrame) -> None:
    fig, ax = plt.subplots(figsize=(4.8, 4.4))
    cmap = plt.get_cmap("viridis")
    n = len(df)
    for i, (_, r) in enumerate(df.iterrows()):
        f = RESULTS / f"scores_{r.representation.replace('/', '_').replace(' ', '_')}.npy"
        if not f.exists():
            continue
        yt, ys = np.load(f)
        fpr, tpr, _ = roc_curve(yt, ys)
        style = "--" if str(r.representation).startswith("Baseline") else "-"
        ax.plot(fpr, tpr, style, lw=1.6, color=cmap(i / max(n - 1, 1)),
                label=f"{r.representation} ({r.auroc:.3f})")
    ax.plot([0, 1], [0, 1], ls=":", color=GREY, lw=1)
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title("ROC curves (test split)")
    ax.legend(fontsize=6.5, loc="lower right")
    fig.tight_layout()
    fig.savefig(FIGURES / "fig4_roc.png", bbox_inches="tight")
    plt.close(fig)
    log.info("wrote fig4_roc.png")




# --------------------------------------------------------------------------- #
# Additional figures required by the report template
# --------------------------------------------------------------------------- #
def fig_dataset_stats() -> None:
    """Length distribution and amino-acid composition by class."""
    from common import AA
    frames = [pd.read_csv(SPLITS / f"{s}.csv") for s in ("train", "val", "test")]
    df = pd.concat(frames, ignore_index=True)

    fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.6))

    for cls, colour in ((0, BLUE), (1, ORANGE)):
        lens = df[df.label == cls].sequence.str.len()
        axes[0].hist(lens, bins=40, alpha=0.6, color=colour,
                     label=CLASS_NAMES[cls])
    axes[0].set_xlabel("Sequence length (residues)")
    axes[0].set_ylabel("Count")
    axes[0].set_title("Length distribution by class")
    axes[0].legend(fontsize=7)

    width = 0.4
    xs = np.arange(len(AA))
    for k, (cls, colour) in enumerate(((0, BLUE), (1, ORANGE))):
        sub = df[df.label == cls].sequence
        counts = np.zeros(len(AA))
        total = 0
        for sq in sub:
            for ch in sq:
                j = AA.find(ch)
                if j >= 0:
                    counts[j] += 1
            total += len(sq)
        axes[1].bar(xs + (k - 0.5) * width, counts / max(total, 1), width,
                    color=colour, label=CLASS_NAMES[cls])
    axes[1].set_xticks(xs)
    axes[1].set_xticklabels(list(AA), fontsize=7)
    axes[1].set_xlabel("Amino acid")
    axes[1].set_ylabel("Frequency")
    axes[1].set_title("Amino-acid composition by class")
    axes[1].legend(fontsize=7)

    fig.tight_layout()
    fig.savefig(FIGURES / "fig5_dataset_stats.png", bbox_inches="tight")
    plt.close(fig)
    log.info("wrote fig5_dataset_stats.png")


def fig_learning_curves() -> None:
    path = RESULTS / "learning_curves.csv"
    if not path.exists():
        log.info("no learning_curves.csv - skipping fig6")
        return
    df = pd.read_csv(path)
    fig, ax = plt.subplots(figsize=(5.4, 3.8))
    cmap = plt.get_cmap("viridis")
    models = list(dict.fromkeys(df.model))
    for i, m in enumerate(models):
        sub = df[df.model == m].sort_values("n_train")
        ls = "--" if "random init" in str(m) else "-"
        ax.plot(sub.n_train, sub.auroc, ls, marker="o", ms=4, lw=1.6,
                color=cmap(i / max(len(models) - 1, 1)), label=m)
    ax.set_xscale("log")
    ax.set_xlabel("Training sequences")
    ax.set_ylabel("Test AUROC")
    ax.set_title("Learning curves")
    ax.grid(alpha=0.25, ls=":")
    ax.legend(fontsize=6.5, loc="lower right")
    fig.tight_layout()
    fig.savefig(FIGURES / "fig6_learning_curves.png", bbox_inches="tight")
    plt.close(fig)
    log.info("wrote fig6_learning_curves.png")


def fig_convergence() -> None:
    path = RESULTS / "convergence_curves.csv"
    if not path.exists():
        log.info("no convergence_curves.csv - skipping fig7")
        return
    df = pd.read_csv(path)
    models = list(dict.fromkeys(df.model))
    n = len(models)
    ncol = min(n, 3)
    nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.4 * ncol, 2.9 * nrow),
                             squeeze=False)
    for ax, m in zip(axes.ravel(), models):
        sub = df[df.model == m].sort_values("epoch")
        ax.plot(sub.epoch, sub.train_loss, "-", lw=1.6, color=BLUE, label="train")
        ax.plot(sub.epoch, sub.val_loss, "-", lw=1.6, color=ORANGE, label="validation")
        ax.set_title(m, fontsize=8)
        ax.set_xlabel("Epoch"); ax.set_ylabel("Log loss")
        ax.grid(alpha=0.25, ls=":")
    for ax in axes.ravel()[n:]:
        ax.axis("off")
    axes[0, 0].legend(fontsize=7)
    fig.suptitle("Probe convergence: train vs. validation log loss", fontsize=10)
    fig.tight_layout()
    fig.savefig(FIGURES / "fig7_convergence.png", bbox_inches="tight")
    plt.close(fig)
    log.info("wrote fig7_convergence.png")


def fig_ablation() -> None:
    path = RESULTS / "ablation_pooling.csv"
    if not path.exists():
        log.info("no ablation_pooling.csv - skipping fig8")
        return
    df = pd.read_csv(path)
    models = list(dict.fromkeys(df.model))
    variants = list(dict.fromkeys(df.variant_label))
    fig, ax = plt.subplots(figsize=(7.2, 3.8))
    width = 0.8 / max(len(variants), 1)
    xs = np.arange(len(models))
    cmap = plt.get_cmap("viridis")
    for k, v in enumerate(variants):
        vals = [df[(df.model == m) & (df.variant_label == v)].auroc.mean()
                for m in models]
        ax.bar(xs + (k - (len(variants) - 1) / 2) * width, vals, width,
               color=cmap(k / max(len(variants) - 1, 1)), label=v)
    ax.set_xticks(xs)
    ax.set_xticklabels(models, fontsize=7, rotation=15, ha="right")
    ax.set_ylabel("Test AUROC")
    ax.set_title("Ablation: pooling strategy and layer choice")
    ax.set_ylim(0.4, 1.0)
    ax.legend(fontsize=7)
    ax.grid(alpha=0.25, ls=":", axis="y")
    fig.tight_layout()
    fig.savefig(FIGURES / "fig8_ablation.png", bbox_inches="tight")
    plt.close(fig)
    log.info("wrote fig8_ablation.png")


def main() -> None:
    set_seed()
    df = load_results()
    for col in ("pretrained", "params", "embed_seconds", "model_key"):
        if col not in df.columns:
            df[col] = np.nan
    df["pretrained"] = df["pretrained"].fillna(False).astype(bool)
    fig_scaling(df)
    fig_embedding_space(df)
    fig_compute(df)
    fig_roc(df)
    fig_dataset_stats()
    fig_learning_curves()
    fig_convergence()
    fig_ablation()
    log.info("figures written to %s", FIGURES)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/make_report.py
"""
Stage 05 - Assemble the written report from the computed results.

Reads dataset_summary.json, results.csv and the figures, and writes
report/report.md with every number filled in. Nothing here is hard-coded, so
rerunning the pipeline on different data regenerates a consistent report.

Usage:
    python src/make_report.py
"""
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parent))
from common import DATA, REPORT, RESULTS, get_logger, load_json

log = get_logger("report")


def fmt(r, m):
    return f"{r[m]:.3f} [{r[f'{m}_lo']:.3f}, {r[f'{m}_hi']:.3f}]"


def main() -> None:
    ds = load_json(DATA / "dataset_summary.json")
    df = pd.read_csv(RESULTS / "results.csv")
    df["pretrained"] = df.get("pretrained", pd.Series([False] * len(df))).fillna(False).astype(bool)

    pre = df[df.pretrained].sort_values("params")
    rnd = df[(~df.pretrained) & (~df.representation.str.startswith("Baseline"))]
    base = df[df.representation.str.startswith("Baseline")]

    smallest, largest = pre.iloc[0], pre.iloc[-1]
    best_base = base.loc[base.auroc.idxmax()] if len(base) else None
    ctrl = rnd.iloc[0] if len(rnd) else None

    gain_8_to_650 = largest.auroc - smallest.auroc
    param_ratio = largest.params / smallest.params
    time_ratio = (largest.embed_seconds / smallest.embed_seconds
                  if pd.notna(largest.get("embed_seconds")) and smallest.get("embed_seconds")
                  else float("nan"))

    # Verdicts derived from confidence-interval overlap, not asserted in advance.
    def overlaps(a, b, m="auroc") -> bool:
        return not (a[f"{m}_lo"] > b[f"{m}_hi"] or b[f"{m}_lo"] > a[f"{m}_hi"])

    scale_verdict = (", a difference whose confidence intervals overlap"
                     if overlaps(smallest, largest)
                     else ", a separation supported by non-overlapping intervals")
    if ctrl is None:
        ctrl_verdict = ""
    elif overlaps(ctrl, smallest):
        ctrl_verdict = (", which is not clearly separated from the pretrained model of "
                        "the same architecture")
    else:
        ctrl_verdict = (", clearly below the pretrained model of identical architecture, "
                        "isolating the contribution of pretraining from that of the "
                        "architecture and pooling")
    if best_base is None:
        base_verdict = ""
    elif best_base.auroc >= largest.auroc:
        base_verdict = (", matching or exceeding every learned representation on this "
                        "task")
    elif overlaps(best_base, largest):
        base_verdict = ", statistically indistinguishable from the largest encoder"
    else:
        base_verdict = ", clearly below the largest encoder"

    rows = []
    for _, r in df.iterrows():
        rows.append(f"| {r.representation} | {int(r['dim'])} | {fmt(r,'accuracy')} | "
                    f"{fmt(r,'macro_f1')} | {fmt(r,'mcc')} | {fmt(r,'auroc')} |")
    table = "\n".join(rows)

    cost_rows = []
    for _, r in pre.iterrows():
        secs = r.get("embed_seconds")
        mem = r.get("peak_gpu_gb")
        cost_rows.append(
            f"| {r.representation} | {int(r.params/1e6)}M | {int(r['dim'])} | "
            f"{'-' if pd.isna(secs) else f'{secs:.0f} s'} | "
            f"{'-' if pd.isna(mem) else f'{mem:.2f} GB'} | {r.auroc:.3f} |")
    cost_table = "\n".join(cost_rows)

    mock_banner = ("\n> **Warning: these numbers come from a `--mock` run on synthetic "
                   "sequences and are not scientific results.** Rerun the pipeline "
                   "without `--mock` before submitting.\n") if ds.get("mock") else ""

    md = f"""# Does scale help? Benchmarking frozen ESM-2 representations for membrane-protein classification

**Course:** CSE 763 Advanced Bioinformatics / CSE 443 Bioinformatics — Term Project
{mock_banner}
## Abstract

Protein language models are routinely used as frozen feature extractors, and larger
checkpoints are generally assumed to give better features. We test that assumption
directly. Four ESM-2 checkpoints spanning {int(smallest.params/1e6)}M to
{int(largest.params/1e6)}M parameters ({param_ratio:.0f}x) were used to embed
{ds['total']} reviewed human proteins, and a logistic-regression probe was trained on
the frozen mean-pooled representations to separate transmembrane from soluble
cytoplasmic proteins. Two classical sequence-composition baselines and a
randomly-initialised encoder of identical architecture serve as controls. Scaling from
{int(smallest.params/1e6)}M to {int(largest.params/1e6)}M parameters changed test AUROC
by {gain_8_to_650:+.3f} ({smallest.auroc:.3f} to {largest.auroc:.3f}){scale_verdict}. The
randomly-initialised control reached {"n/a" if ctrl is None else f"{ctrl.auroc:.3f}"}
AUROC{ctrl_verdict}. The strongest composition baseline reached
{"n/a" if best_base is None else f"{best_base.auroc:.3f}"} AUROC{base_verdict}.

## 1. Introduction

Transformer encoders trained on large protein sequence corpora with a masked-language
objective learn representations that transfer to structural and functional prediction
tasks. ESM-2 is released as a family of checkpoints that share a training corpus,
tokenisation and objective and differ almost only in depth and width, which makes it an
unusually clean testbed for a scaling study.

Practitioners face a concrete decision: is the largest checkpoint that fits in memory
always the right default? Larger encoders cost more time and memory at inference, and
for a downstream task that may be close to linearly separable, the extra capacity may
buy very little. This project quantifies that trade-off on a single well-defined task.

We deliberately keep the encoders **frozen** and fit only a linear probe. Fine-tuning
confounds representation quality with optimisation behaviour: a large model may
underperform simply because it is harder to fine-tune on a small dataset. A linear probe
measures how much task-relevant information is linearly decodable from the
representation, which is the quantity relevant to the "frozen feature extractor" use
case, and it makes every row of the results table comparable under an identical
classifier.

**Research questions.**
1. How does linear-probe performance scale with ESM-2 encoder size on this task?
2. How much of the performance is attributable to pretraining, as opposed to the
   architecture and mean pooling?
3. Do the learned representations beat simple amino-acid and k-mer composition
   features, which are known to be informative for membrane proteins?
4. Is the additional compute of the largest checkpoint justified by its accuracy?

## 1.1 Related work

The expectation that scale improves representations comes from the ESM line of work:
Rives et al. [2] showed biochemical and structural information emerges from
masked-language pretraining at scale, and Lin et al. [1] showed atomic-resolution
structural information emerges as models are scaled to 15B parameters. Both establish
scaling benefits for the pretraining objective and for structure prediction, which is
closely aligned with what the objective learns.

Whether this transfers to arbitrary downstream tasks is a separate question, and the
evidence is mixed. Li et al. [3] conducted 370 transfer-learning experiments and found
that while nearly all downstream tasks benefit from pretraining relative to naive
representations, performance for most tasks does not scale with model size or
pretraining time, relying instead on low-level features acquired early in pretraining.
Independent work [4] found larger models do not necessarily outperform smaller ones,
particularly under limited labelled data, and additionally reported that mean pooling
outperformed other embedding-compression strategies. FLIP2 [6] reported that simpler
models frequently matched or exceeded fine-tuned protein language models.

The value of simple baselines is itself an established argument: Shanehsazzadeh et al.
[5] showed small supervised models can match pretrained TAPE models at a fraction of
the compute, and argued that simple baselines are necessary to interpret pretrained
model results. Detlefsen et al. [7] showed that minor methodological differences
produce substantially different representations, motivating an identical evaluation
protocol across all conditions.

For the task itself, DeepTMHMM [8] is the reference method for transmembrane topology
prediction, and DeepLoc 2.1 [9] constructs comparable membrane/soluble labels from
UniProt, though with a stricter experimental-assertion filter than we apply here.

**Positioning.** This study does not claim a novel research question. It is a
small-scale, independent replication of the scaling conclusion in [3], on a downstream
task absent from that suite, with a fully reproducible protocol, explicit compute
accounting, and confidence intervals on every reported number.

## 2. Data

Sequences were retrieved from the UniProt REST API, restricted to reviewed
(Swiss-Prot) entries from *Homo sapiens*.

- **Positive class (transmembrane):** entries carrying keyword KW-0812.
- **Negative class (soluble cytoplasmic):** entries with subcellular location
  Cytoplasm (SL-0091) that carry neither KW-0472 (Membrane) nor KW-0812.

Exact queries are recorded in `data/dataset_summary.json`.

**Filtering.** Sequences were restricted to {ds['length_window'][0]}-{ds['length_window'][1]}
residues, well within ESM-2's 1024-token pretraining context. Entries containing
non-standard residues (X, B, Z, U, O) were discarded, as were exact duplicates.

**Redundancy reduction.** Homologous proteins split across train and test would inflate
performance. We applied a greedy filter over the *combined* set: sequences were
processed longest-first and a sequence was dropped if its {ds['kmer_k']}-mer Jaccard
similarity to any already-retained sequence exceeded {ds['jaccard_threshold']}. This is a
cheap approximation to CD-HIT clustering; it removes near-duplicates but does not
guarantee family-level separation (see Limitations).

**Final dataset.** {ds['per_class']} sequences per class ({ds['total']} total), mean length
{ds['mean_length']:.0f} residues, split 70/15/15 in a stratified manner:
{ds['splits']['train']} train / {ds['splits']['val']} validation / {ds['splits']['test']} test.
The balanced design means the majority-class baseline is 0.500 accuracy and 0.500 AUROC.

## 3. Methods

### 3.1 Representations

| Checkpoint | Layers | Hidden dim |
|---|---|---|
{chr(10).join(f"| {r.representation} | {int(r.layers)} | {int(r['dim'])} |" for _, r in pre.iterrows())}

Each sequence was passed once through the frozen encoder. The final hidden states were
mean-pooled over real residue positions only: padding, `<cls>` and `<eos>` were masked
out before averaging, since including them mixes non-residue information into the vector
and makes the representation length-dependent.

### 3.2 Controls and baselines

- **AAC + length.** The 20 amino-acid frequencies plus log sequence length. Membrane
  proteins are hydrophobic-enriched, so this baseline is expected to be strong and is
  the honest bar a language model must clear.
- **3-mer frequency.** Normalised counts over all 8000 tripeptides — a bag-of-k-mers
  representation with no notion of order beyond the window.
- **Randomly-initialised ESM-2 8M.** Identical architecture, tokeniser and pooling, but
  no pretrained weights. Any performance here reflects what the architecture plus mean
  pooling extracts for free; the gap to the pretrained 8M model is the value added by
  pretraining. This control is frequently omitted in comparisons of this kind.

### 3.3 Probe protocol

Features were standardised with training-split statistics. The L2 strength C of a
logistic regression was swept over {load_json(RESULTS / 'probe_config.json')['C_grid']}
and chosen by validation macro-F1; the model was then refit on train + validation with
the selected C and evaluated once on the test split. We report accuracy, macro-F1,
Matthews correlation coefficient and AUROC, each with a 95% percentile bootstrap
confidence interval over
{load_json(RESULTS / 'probe_config.json')['n_bootstrap']} resamples of the
{load_json(RESULTS / 'probe_config.json')['n_test']} test proteins. MCC is included
because it stays informative if the class balance shifts, and AUROC because it is
threshold-independent.

## 4. Results

| Representation | Dim | Accuracy | Macro-F1 | MCC | AUROC |
|---|---|---|---|---|---|
{table}

*Values are point estimates with 95% bootstrap confidence intervals.*

![Scaling](../figures/fig1_scaling.png)

**Figure 1.** AUROC and macro-F1 against encoder size. Dashed grey lines mark the
composition baselines; the red cross is the randomly-initialised control.

![Embedding space](../figures/fig2_embedding_space.png)

**Figure 2.** Two-dimensional projection of test-set embeddings, coloured by class, for
the smallest and largest pretrained checkpoints.

![Compute](../figures/fig3_compute.png)

**Figure 3.** Test AUROC against the wall-clock time to embed the full dataset.

![ROC](../figures/fig4_roc.png)

**Figure 4.** ROC curves for all representations on the test split.

### 4.1 Scaling

Moving from {int(smallest.params/1e6)}M to {int(largest.params/1e6)}M parameters changed
AUROC by {gain_8_to_650:+.3f}, a {param_ratio:.0f}-fold increase in parameters. Whether
this gain is meaningful should be judged against the bootstrap intervals in the table: if
the intervals for the smallest and largest models overlap substantially, the dataset does
not support a claim that scale helps on this task.

### 4.2 Cost of scale

| Model | Params | Dim | Embed time | Peak GPU | AUROC |
|---|---|---|---|---|---|
{cost_table}

The largest checkpoint takes roughly {"n/a" if np.isnan(time_ratio) else f"{time_ratio:.1f}x"}
as long to embed the dataset as the smallest.

### 4.3 Value of pretraining

The randomly-initialised control reached
{"n/a" if ctrl is None else f"AUROC {ctrl.auroc:.3f}"} against
{smallest.auroc:.3f} for the pretrained model of the same architecture. The difference
isolates the contribution of masked-language pretraining from the architecture and the
pooling operation.

### 4.4 Against classical features

The strongest composition baseline was
{"n/a" if best_base is None else f"{best_base.representation} at AUROC {best_base.auroc:.3f}"}.
A learned representation is only worth its cost if it clears this bar by a margin larger
than the confidence intervals — an important check, because membrane-protein
classification is a task where hydrophobicity alone carries a great deal of signal.

## 5. Discussion

Three points follow from the table.

First, the relationship between scale and downstream performance is task-dependent, not
universal. On a task where the decision boundary is close to linear in composition
space, small checkpoints can capture most of the available signal, and the marginal
return on parameters is compressed by a ceiling effect.

Second, the random-init control matters. Mean-pooled representations from an untrained
transformer are not meaningless: the architecture and the pooling operation already
encode something akin to composition statistics. Any comparison that omits this control
risks attributing to pretraining what is in fact free.

Third, benchmark tables that report only accuracy hide the operational trade-off. A
model that is marginally better but several times slower may be the wrong default for a
proteome-scale screen, and the right default for a small high-stakes set.

## 6. Limitations

- **Homology.** The k-mer Jaccard filter removes near-duplicates but not remote
  homologues. CD-HIT or MMseqs2 clustering at 30% identity with cluster-disjoint splits
  would be stricter, and performance would likely drop somewhat under that protocol.
- **Label noise.** UniProt keyword annotations are partly inferred rather than
  experimentally verified, so a fraction of labels is unreliable.
- **One task, one organism.** Results on human membrane-protein classification need not
  transfer to contact prediction, thermostability, or non-eukaryotic proteomes.
- **Frozen only.** Fine-tuning could change the ranking; larger models often gain more
  from it. This study bounds the frozen-feature regime only.
- **Mean pooling only.** Alternatives such as `<cls>` pooling, attention pooling or
  per-residue probes could favour different checkpoints.
- **Single seed.** The probe is deterministic given the split, but the split itself was
  drawn once; repeating over several splits would give a better variance estimate.

## 7. Reproducibility

```bash
python src/fetch_data.py --per-class {ds['per_class']}
python src/embed.py
python src/probe.py
python src/figures.py
python src/make_report.py
```

Seed {42} is fixed throughout. All intermediate artefacts (splits, embeddings, timings,
per-representation test scores) are written to `data/` and `results/`.

## References

1. Lin Z. et al. Evolutionary-scale prediction of atomic-level protein structure with a
   language model. *Science* 379(6637):1123-1130 (2023). doi:10.1126/science.ade2574
2. Rives A. et al. Biological structure and function emerge from scaling unsupervised
   learning to 250 million protein sequences. *PNAS* 118(15):e2016239118 (2021).
   doi:10.1073/pnas.2016239118
3. Li F.-Z., Amini A. P., Yue Y., Yang K. K., Lu A. X. Feature Reuse and Scaling:
   Understanding Transfer Learning with Protein Language Models. *ICML 2024*,
   PMLR 235:27351-27375. doi:10.1101/2024.02.05.578959
4. Medium-sized protein language models perform well at transfer learning on realistic
   datasets. *Scientific Reports* (2025). doi:10.1038/s41598-025-05674-x
5. Shanehsazzadeh A., Belanger D., Dohan D. Is Transfer Learning Necessary for Protein
   Landscape Prediction? arXiv:2011.03443 (2020). MLSB workshop, NeurIPS 2020.
6. Dallago C. et al. FLIP: Benchmark tasks in fitness landscape inference for proteins.
   *NeurIPS Datasets and Benchmarks* 34:26601-26622 (2021).
7. Detlefsen N. S., Hauberg S., Boomsma W. Learning meaningful representations of
   protein sequences. *Nature Communications* 13:1914 (2022).
   doi:10.1038/s41467-022-29443-w
8. Hallgren J. et al. DeepTMHMM predicts alpha and beta transmembrane proteins using
   deep neural networks. *bioRxiv* (2022). doi:10.1101/2022.04.08.487609
9. DeepLoc 2.1: multi-label membrane protein type prediction using protein language
   models. *Nucleic Acids Research* Web Server issue (2024). PMC11223819
10. Steinegger M., Soding J. MMseqs2 enables sensitive protein sequence searching for
    the analysis of massive data sets. *Nature Biotechnology* 35:1026-1028 (2017).
11. The UniProt Consortium. UniProt: the Universal Protein Knowledgebase in 2025.
    *Nucleic Acids Research* 53(D1):D609-D617 (2025). doi:10.1093/nar/gkae1010
"""

    (REPORT / "report.md").write_text(md)
    log.info("wrote %s", REPORT / "report.md")


if __name__ == "__main__":
    main()


## 2. Build the dataset

Downloads reviewed human proteins from UniProt, filters them by length and
alphabet, removes near-duplicates with a k-mer Jaccard filter, balances the
classes and writes stratified 70/15/15 splits.

Reduce `--per-class` if you want a faster run; 1500 is a reasonable default.

In [ ]:
!python src/fetch_data.py --per-class 1500


In [ ]:
import pandas as pd, json
summary = json.load(open('data/dataset_summary.json'))
print(json.dumps(summary, indent=2))
train = pd.read_csv('data/splits/train.csv')
print(train.head())
print('\nclass balance in train:'); print(train.label.value_counts())


## 3. Extract frozen embeddings

Every sequence passes once through each frozen encoder and is mean-pooled over
real residue positions. Wall-clock time and peak GPU memory are recorded.

If the 650M model runs out of memory, rerun this cell with a smaller
`--max-tokens` (for example `--max-tokens 8192`).

In [ ]:
!python src/embed.py --max-tokens 8192


## 4. Train the probes

Standardise, sweep the regularisation strength on the validation split, refit on
train + validation, evaluate once on test with bootstrap confidence
intervals.

In [ ]:
!python src/probe.py


In [ ]:
import pandas as pd
from IPython.display import Markdown, display
display(Markdown(open('results/results.md').read()))


## 4b. Ablation study, learning curves and convergence

Swaps the pooling strategy and layer choice, removes pipeline components one at a
time, and traces performance against training-set size and training epoch. All of
this reuses the cached embeddings, so it costs no extra GPU time.

In [ ]:
!python src/ablation.py


In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('results/ablation.md').read()))
display(Markdown(open('results/hyperparameter_sweep.md').read()))


## 5. Figures

In [ ]:
!python src/figures.py


In [ ]:
from IPython.display import Image, display
figs = ['fig1_scaling', 'fig2_embedding_space', 'fig3_compute', 'fig4_roc',
        'fig5_dataset_stats', 'fig6_learning_curves', 'fig7_convergence', 'fig8_ablation']
for f in figs:
    display(Image(f'figures/{f}.png'))


## 6. Report

In [ ]:
!python src/make_report.py


In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('report/report.md').read()))


## 7. Download everything

Produces a single zip containing the splits, embeddings metadata, results table,
figures and the filled-in report.

In [ ]:
!zip -qr esm2_benchmark_outputs.zip results figures report data/dataset_summary.json data/emb/timings.json data/splits
print('zip written')
try:
    from google.colab import files
    files.download('esm2_benchmark_outputs.zip')
except Exception as e:
    print('not running in Colab, download manually:', e)
